In [2]:
import numpy as np
import pandas as pd
import joblib

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

data = np.load("data/processed.npz")

X_train = data["X_train"].reshape(len(data["X_train"]), -1)
X_val = data["X_val"].reshape(len(data["X_val"]), -1)

y_train = data["y_train"].ravel()
y_val = data["y_val"].ravel()

y_scaler = joblib.load("data/target_scaler.pkl")

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    # "Random Forest Regressor": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    # "Extra Trees Regressor": ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    # "Gradient Boosting Regressor": GradientBoostingRegressor(n_estimators=100, random_state=42),
    # "Decision Tree Regressor": DecisionTreeRegressor(random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)

    train_mse = mean_squared_error(y_train, train_pred)
    val_mse = mean_squared_error(y_val, val_pred)

    train_actual = y_scaler.inverse_transform(y_train.reshape(-1, 1)).ravel()
    val_actual = y_scaler.inverse_transform(y_val.reshape(-1, 1)).ravel()
    train_pred_actual = y_scaler.inverse_transform(train_pred.reshape(-1, 1)).ravel()
    val_pred_actual = y_scaler.inverse_transform(val_pred.reshape(-1, 1)).ravel()

    results.append({
        "Model": name,
        "Train MSE": train_mse,
        "Val MSE": val_mse,
        "Train RMSE": np.sqrt(mean_squared_error(train_actual, train_pred_actual)),
        "Val RMSE": np.sqrt(mean_squared_error(val_actual, val_pred_actual)),
        "Train MAE": mean_absolute_error(train_actual, train_pred_actual),
        "Val MAE": mean_absolute_error(val_actual, val_pred_actual)
    })

results_df = pd.DataFrame(results).sort_values("Val RMSE")
display(results_df.round(4))

,Model,Train MSE,Val MSE,Train RMSE,Val RMSE,Train MAE,Val MAE
1,Ridge Regression,0.2615,0.4213,54.7659,69.5154,34.3289,45.6548
0,Linear Regression,0.2455,0.5270,53.0640,77.7503,35.4229,54.9473


In [ ]:
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

models = {
    "LightGBM": LGBMRegressor(n_estimators=100, device_type="gpu", random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, tree_method="hist", device="cuda", random_state=42),
    "CatBoost": CatBoostRegressor(iterations=100, task_type="GPU", devices="0", verbose=False, random_seed=42)
}